# FlashEats — Class 8
## Building a Simple, Dependable Data Pipeline

### The story

Yesterday, our FlashEats workflow model worked.

We could calculate:

- Late Delivery Rate
- Order-to-Pickup Time
- Transit Time
- Customer interaction signals
- Intervention metrics

Today the client asks:

> **“Great. Can this run every morning without you?”**

That changes the engineering problem.

A successful notebook run is not the same thing as a dependable pipeline.

Today's goal:

```text
ONE COMMAND
    ↓
EXTRACT
    ↓
VALIDATE
    ↓
CLEAN
    ↓
TRANSFORM
    ↓
SAVE
    ↓
LOG
```

and if something is unsafe:

```text
FAIL CLEARLY
+
DO NOT PUBLISH BAD OUTPUT
```

In [ ]:
!pip -q install pandas requests flask

In [ ]:
# SETUP — local first, Colab upload only if necessary.
# No matplotlib is required for this walkthrough.

import json
import os
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import pandas as pd


def find_pack_root():
    cwd = Path.cwd()

    # Fast path: notebook is already inside the pack.
    if (cwd / "database" / "flasheats.db").exists():
        return cwd

    # Common folder names.
    for name in [
        "FlashEats_Classroom_Pack_V2",
        "FlashEats_Classroom_Pack",
    ]:
        candidate = cwd / name
        if (candidate / "database" / "flasheats.db").exists():
            return candidate

    # Search locally, then Colab.
    for root in [cwd, cwd.parent, Path("/content")]:
        if not root.exists():
            continue

        for candidate in root.rglob("*"):
            if (
                candidate.is_dir()
                and (candidate / "database" / "flasheats.db").exists()
            ):
                return candidate

    return None


BASE = find_pack_root()

if BASE is None:
    try:
        from google.colab import files

        print("Upload FlashEats_Class8_Classroom_Pack.zip")
        uploaded = files.upload()
        zip_name = next(
            name for name in uploaded
            if name.endswith(".zip")
        )

        extract_dir = Path("/content/flasheats_class8")
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall(extract_dir)

        BASE = find_pack_root()

    except Exception as exc:
        print("Automatic setup failed:", exc)

if BASE is None:
    raise FileNotFoundError(
        "Could not locate a FlashEats pack containing database/flasheats.db"
    )

PROJECT = BASE / "Class8_Project"

print("Pack:", BASE)
print("Project:", PROJECT)
print("Database exists:", (BASE / "database" / "flasheats.db").exists())
print("run_pipeline.py exists:", (PROJECT / "run_pipeline.py").exists())

# 1. Working once ≠ dependable

Imagine yesterday's notebook contains:

```python
orders = load_orders()
dispatch = fetch_dispatch()
journey = build_journey(orders, dispatch)
journey.to_csv("order_journey.csv")
```

It works today.

## What assumptions are hidden?


1. What if the API returns only page 1?
2. What if the API returns HTTP 500?
3. What if `promised_eta` disappears?
4. What if the source is stale?
5. What if the job is run twice?
6. What if it crashes halfway through?

### Deduction

> **Dependability means making important assumptions explicit and testable.**

# 2. Turn the notebook into stages

We will use five explicit stages:

```text
EXTRACT → VALIDATE → CLEAN → TRANSFORM → SAVE
```

Why stages?

Because when something fails, we want to know **what responsibility failed**.

Open the actual project structure:

In [ ]:
for path in sorted(PROJECT.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT))

### Discussion

- Why should validation happen before transformation?
- Why preserve raw API responses?
- Why is `save()` a separate stage?
- Would you put every operation in one giant function?

### Deduction

> Functions are not the goal. **Isolation of responsibility** is the goal.

# 3. Configuration should not be buried in code

Bad:

```python
API_URL = "http://localhost:8000"
MAX_RETRIES = 3
OUTPUT = "/Users/me/Desktop/output.csv"
```

Better:

```text
Code = behavior
Environment/config = where and how it runs
```

Let's inspect the example configuration.

In [ ]:
print((PROJECT / "config" / ".env.example").read_text())

### Discussion

1. Which values could differ between local, staging and production?
2. Which values should never be committed if they are secrets?
3. Why is `MAX_RETRIES` configuration but retry logic code?

### Deduction

> A pipeline is easier to move between environments when deployment-specific values are outside core business logic.

# 4. Validation becomes a gate

Class 6 taught us how to discover data-quality rules.

Class 8 changes the question:

> **How do we make those rules execute every run?**

Our pipeline checks:

- required columns,
- duplicate order IDs,
- critical missing completion timestamps,
- freshness,
- required Dispatch fields.

Let's inspect the validation module.

In [ ]:
validation_code = (PROJECT / "pipeline" / "validate.py").read_text()

print(validation_code[:5000])

### Story

Suppose `promised_eta` disappears tomorrow.

A weak pipeline continues until some later line crashes — or worse, produces a misleading output.

A dependable pipeline should say:

```text
VALIDATION FAILED
orders: missing required columns: ['promised_eta']

No processed output written.
```

### Deduction

> **Validation protects downstream consumers from silently bad output.**

# 5. Retries: retry the right failures

The FlashEats Dispatch API deliberately behaves like a real external dependency:

- page 3 fails once with HTTP 500,
- page 5 rate-limits once with HTTP 429,
- pagination must be completed.

A retry is appropriate for a **transient** failure.

A retry is usually not useful for:

- missing required column,
- invalid credentials,
- incompatible schema,
- a business-rule failure.

Let's inspect the retry logic.

In [ ]:
extract_code = (PROJECT / "pipeline" / "extract.py").read_text()

start = extract_code.find("def _request_page")
end = extract_code.find("def extract_dispatch_api")

print(extract_code[start:end])

### Discussion

1. Why limit the number of retries?
2. Why wait longer between repeated attempts?
3. Why should HTTP 429 and HTTP 500 be treated differently from a schema failure?
4. Could retrying a write operation be dangerous?

### Deduction

> **Retry transient failures; fail fast on conditions that require human or code changes.**

# 6. Idempotency: what happens if we run twice?

A job that appends the same 1,600 orders every time is repeatable — but not dependable.

For this class:

```text
same logical run_date
        ↓
same output partition
        ↓
atomically replace output
```

Let's inspect the save stage.

In [ ]:
save_code = (PROJECT / "pipeline" / "save.py").read_text()
print(save_code)

### Discussion

1. Why partition outputs by logical run date?
2. Why replace instead of append for this pipeline?
3. When would append actually be correct?
4. Why write a temporary file before replacing the final file?

### Deduction

> **A rerun should not create unintended duplicate effects.**

# 7. Logging should help the next engineer

Bad log:

```text
ERROR
```

Useful log:

```text
Dispatch API failure
page=3
status=500
attempt=2/3
```

A log should help answer:

- which stage?
- which source?
- how many records?
- which attempt?
- what failed?
- did output get published?

The pipeline writes both console and file logs.

In [ ]:
print((PROJECT / "pipeline" / "logging_utils.py").read_text())

### Deduction

> **A dependable pipeline is diagnosable, not merely executable.**

# 8. Run the complete flow with one command

This is the handoff moment.

From `Class8_Project`:

```bash
python run_pipeline.py --run-date 2026-09-22
```

The command should:

1. start the mock API if needed,
2. extract local sources,
3. retrieve every Dispatch API page,
4. retry transient failures,
5. preserve raw pages,
6. validate inputs,
7. clean the agreed business grain,
8. build the Class 7 order journey,
9. save metrics + validation report,
10. write logs.

Run it from this notebook:

In [ ]:
# Install only the small dependencies used by the project.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")],
    check=True,
)

result = subprocess.run(
    [
        sys.executable,
        str(PROJECT / "run_pipeline.py"),
        "--run-date",
        "2026-09-22",
    ],
    cwd=PROJECT,
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("\nSTDOUT:\n", result.stdout[-5000:])
print("\nSTDERR:\n", result.stderr[-3000:])

# 9. Prove idempotency

Run exactly the same logical job again.

What should happen?

- output row count should not double,
- the same partition should be replaced,
- the final business dataset should remain one row per order.

In [ ]:
output_path = (
    PROJECT
    / "data"
    / "processed"
    / "run_date=2026-09-22"
    / "order_journey.csv"
)

first = pd.read_csv(output_path)
first_rows = len(first)

second_run = subprocess.run(
    [
        sys.executable,
        str(PROJECT / "run_pipeline.py"),
        "--run-date",
        "2026-09-22",
    ],
    cwd=PROJECT,
    text=True,
    capture_output=True,
)

second = pd.read_csv(output_path)
second_rows = len(second)

print("First run rows:", first_rows)
print("Second run rows:", second_rows)
print("Rows doubled?", second_rows == first_rows * 2)
print("Same logical row count?", first_rows == second_rows)

### Deduction

> **“Can rerun” and “safe to rerun” are different properties.**

# 10. Controlled failure: break the schema

A dependable pipeline should not just succeed nicely.

It should fail nicely.

Run:

In [ ]:
failure = subprocess.run(
    [
        sys.executable,
        str(PROJECT / "run_pipeline.py"),
        "--run-date",
        "2026-09-22",
        "--chaos",
        "missing_column",
    ],
    cwd=PROJECT,
    text=True,
    capture_output=True,
)

print("RETURN CODE:", failure.returncode)
print(failure.stdout[-3000:])

### Discuss

1. Did the pipeline explain what was wrong?
2. Did it keep retrying a non-retryable problem?
3. Did it publish a new processed dataset?
4. What would an on-call engineer need next?

### Deduction

> **Dependability includes safe, understandable failure.**

# 11. Gate 2 — Data readiness review

The final output of this module is not merely `pipeline.py`.

We need to be able to answer:

> **Is the data foundation ready for the next product/AI phase?**

Review the template:

In [ ]:
print((PROJECT / "GATE2_DATA_READINESS_TEMPLATE.md").read_text())

# The story to carry out of Class 8

We started with:

```text
“Here is a notebook that works.”
```

We ended with:

```text
ONE COMMAND
   ↓
KNOWN CONFIGURATION
   ↓
BOUNDED RETRIES
   ↓
VALIDATION GATES
   ↓
EXPLICIT TRANSFORMS
   ↓
IDEMPOTENT OUTPUT
   ↓
USEFUL LOGS
   ↓
READINESS DECISION
```

The core lesson:

> **A dependable pipeline is not one that never fails. It produces trustworthy output when things are healthy, and fails clearly and safely when they are not.**

# Knowledge check

## Poll 1
The Dispatch API returns HTTP 500 once. What should the pipeline usually do?

A. Delete the input  
B. Retry a bounded number of times  
C. Remove validation  
D. Retry forever


---

## Poll 2
`promised_eta` disappears from the source schema. What should happen?

A. Retry the API  
B. Fill the column with NULL and publish  
C. Fail the validation gate with a useful message  
D. Ignore it because yesterday's run worked


---

## Poll 3
A pipeline run for the same `run_date` is executed twice. Which behavior best demonstrates idempotency?

A. Double the rows  
B. Create two identical permanent copies  
C. Replace the same logical output without duplicate effects  
D. Refuse every rerun


---

## Poll 4
Which is the best log message?

A. `ERROR`  
B. `Something broke`  
C. `Dispatch page=5 status=429 attempt=2/3 wait=1s`  
D. `Try again`


---

## Poll 5
Which Class 6 concept changes meaningfully in Class 8?

A. We stop validating data  
B. Data-quality rules become executable gates on every run  
C. We replace business rules with infrastructure  
D. We remove source ownership